# NLP

NLP, ou Processamento de Linguagem Natural, e uma area da ciencia de dados que estuda como transformar textos em informacao estruturada. Com essas tecnicas, conseguimos analisar noticias, identificar temas, contar palavras, comparar documentos e preparar dados textuais para modelos de aprendizado de maquina.

In [5]:
import pandas as pd

df = pd.read_csv("noticias_lbf_1000.csv", sep=";")
df.head()

,titulo_noticia,time_mais_citado,qtd_citacoes_time_mais_citado,qtd_total_palavras,qtd_times_citados,link
0,retorno saboroso cat,Sport/Uninassau/Inst. Todos,10,780,2,https://lbf.com.br/noticias/retorno-saboroso-cat/
1,brasil e prata na americup 3x3,Nenhum time identificado,0,200,0,https://lbf.com.br/noticias/brasil-e-prata-na-...
2,tecnica do maranhao basquete quer time guerrei...,Sport/Uninassau/Inst. Todos,1,593,1,https://lbf.com.br/noticias/tecnica-do-maranha...
3,contando com dupla de pivos brasileiras chicag...,Sport/Uninassau/Inst. Todos,1,372,1,https://lbf.com.br/noticias/contando-com-dupla...
4,bradesco derrota o xv mesmo atuando fora de ca...,Sport/Uninassau/Inst. Todos,1,303,1,https://lbf.com.br/noticias/bradesco-derrota-o...


## Trabalhando apenas com o texto

Por enquanto, vamos trabalhar somente com a coluna `texto`, que contem o corpo completo de cada noticia. As outras colunas, como `titulo`, `descricao`, `tags` e `data`, continuam no `DataFrame`, mas vamos deixar elas de lado neste primeiro momento para entender melhor o conteudo textual.

## Passos da analise

Vamos preparar os textos aos poucos:

1. Limpar os textos.
2. Remover palavras muito comuns.
3. Criar uma representacao Bag of Words.
4. Contar palavras frequentes.
5. Transformar textos em numeros para analises posteriores.

## 1. Limpeza basica

Na celula abaixo:

- `wordpunct_tokenize` separa o texto em palavras e pontuacao.
- `texto.lower()` coloca tudo em minusculas.
- `unidecode(texto)` troca letras acentuadas por letras sem acento.
- `token.isalnum()` mantem apenas letras e numeros.
- `" ".join(tokens)` junta os tokens em uma frase limpa.
- `.apply(limpar_texto)` aplica a funcao em todas as noticias.

Exemplo: `"Olá, Senado!"` vira `"ola senado"`.

In [11]:
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode

def limpar_texto(texto):
    texto = texto.lower()
    texto = unidecode(texto)
    tokens = wordpunct_tokenize(texto)
    tokens = [token for token in tokens if token.isalnum()]
    return " ".join(tokens)

df["titulo_limpo"] = df["titulo_noticia"].apply(limpar_texto)

df[["titulo_noticia", "titulo_limpo"]].head()

,titulo_noticia,titulo_limpo
0,retorno saboroso cat,retorno saboroso cat
1,brasil e prata na americup 3x3,brasil e prata na americup 3x3
2,tecnica do maranhao basquete quer time guerrei...,tecnica do maranhao basquete quer time guerrei...
3,contando com dupla de pivos brasileiras chicag...,contando com dupla de pivos brasileiras chicag...
4,bradesco derrota o xv mesmo atuando fora de ca...,bradesco derrota o xv mesmo atuando fora de ca...


## 2. Removendo stopwords

Stopwords sao palavras muito comuns, como `a`, `o`, `de`, `para` e `que`. Elas aparecem muito, mas geralmente ajudam pouco a entender o tema de um texto.

Na celula abaixo:

- `stopwords.words("portuguese")` carrega stopwords em portugues.
- `texto.split()` separa o texto limpo em palavras.
- `token not in stopwords_pt` remove as palavras muito comuns.
- `.str.join(" ")` junta os tokens restantes em um texto sem stopwords.
- `.apply(remover_stopwords)` aplica a funcao em todas as noticias.

Exemplo: `"o senador falou com a imprensa"` vira `["senador", "falou", "imprensa"]`.

In [12]:

import nltk
from nltk.corpus import stopwords
from unidecode import unidecode

nltk.download("stopwords", quiet=True)

stopwords_pt = stopwords.words("portuguese")
stopwords_pt = [unidecode(palavra) for palavra in stopwords_pt]
stopwords_pt = set(stopwords_pt)

def remover_stopwords(texto):
    tokens = texto.split()
    tokens = [token for token in tokens if token not in stopwords_pt]
    return tokens

df["tokens_titulo_sem_stopwords"] = df["titulo_limpo"].apply(remover_stopwords)
df["titulo_sem_stopwords"] = df["tokens_titulo_sem_stopwords"].str.join(" ")

df[["titulo_limpo", "tokens_titulo_sem_stopwords", "titulo_sem_stopwords"]].head()

,titulo_limpo,tokens_titulo_sem_stopwords,titulo_sem_stopwords
0,retorno saboroso cat,"[retorno, saboroso, cat]",retorno saboroso cat
1,brasil e prata na americup 3x3,"[brasil, prata, americup, 3x3]",brasil prata americup 3x3
2,tecnica do maranhao basquete quer time guerrei...,"[tecnica, maranhao, basquete, quer, time, guer...",tecnica maranhao basquete quer time guerreiro ...
3,contando com dupla de pivos brasileiras chicag...,"[contando, dupla, pivos, brasileiras, chicago,...",contando dupla pivos brasileiras chicago super...
4,bradesco derrota o xv mesmo atuando fora de ca...,"[bradesco, derrota, xv, atuando, casa, paulist...",bradesco derrota xv atuando casa paulista femi...


## 3. Bag of Words

No Bag of Words, cada linha representa uma noticia e cada coluna representa uma palavra. O valor indica quantas vezes aquela palavra apareceu na noticia.

Exemplo: `"senador falou senador"` teria `senador = 2` e `falou = 1`.

In [15]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
matriz_bow = vectorizer.fit_transform(df["titulo_sem_stopwords"])

df_bow = pd.DataFrame(
    matriz_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

df_bow.head()

,04,05,06,07,08,10,100,100paio,100si,100x37,...,web,wheeler,wnba,workshop,xaras,xv,yasmim,yuli,zafer,zanon
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


### Removendo colunas com numeros

Agora vamos remover qualquer coluna cujo nome tenha pelo menos um numero.

In [16]:
colunas_com_numeros = [col for col in df_bow.columns if any(char.isdigit() for char in col)]

df_bow = df_bow.drop(columns=colunas_com_numeros)

print(f"{len(colunas_com_numeros)} colunas removidas")
df_bow.head()

71 colunas removidas


,ab,abc,abcd,aberta,abertas,aberto,abertos,abertura,abre,abrem,...,web,wheeler,wnba,workshop,xaras,xv,yasmim,yuli,zafer,zanon
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


### Criando o DataFrame final

Agora vamos juntar os metadados das noticias com as colunas do Bag of Words. Para evitar conflito de nomes, as colunas do Bag of Words recebem o prefixo `bow_`.

In [17]:
metadados = df[[
    "titulo_noticia",
    "time_mais_citado",
    "qtd_citacoes_time_mais_citado",
    "qtd_total_palavras",
    "qtd_times_citados",
    "link"
]].reset_index(drop=True)

bow_com_prefixo = df_bow.add_prefix("bow_").reset_index(drop=True)

df_final = pd.concat([metadados, bow_com_prefixo], axis=1)

df_final.head()

,titulo_noticia,time_mais_citado,qtd_citacoes_time_mais_citado,qtd_total_palavras,qtd_times_citados,link,bow_ab,bow_abc,bow_abcd,bow_aberta,...,bow_web,bow_wheeler,bow_wnba,bow_workshop,bow_xaras,bow_xv,bow_yasmim,bow_yuli,bow_zafer,bow_zanon
0,retorno saboroso cat,Sport/Uninassau/Inst. Todos,10,780,2,https://lbf.com.br/noticias/retorno-saboroso-cat/,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,brasil e prata na americup 3x3,Nenhum time identificado,0,200,0,https://lbf.com.br/noticias/brasil-e-prata-na-...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,tecnica do maranhao basquete quer time guerrei...,Sport/Uninassau/Inst. Todos,1,593,1,https://lbf.com.br/noticias/tecnica-do-maranha...,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,contando com dupla de pivos brasileiras chicag...,Sport/Uninassau/Inst. Todos,1,372,1,https://lbf.com.br/noticias/contando-com-dupla...,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,bradesco derrota o xv mesmo atuando fora de ca...,Sport/Uninassau/Inst. Todos,1,303,1,https://lbf.com.br/noticias/bradesco-derrota-o...,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


## Exemplo de analise

Agora podemos fazer uma analise simples das palavras do Bag of Words: quais aparecem mais, quais aparecem menos e quantas palavras diferentes temos no vocabulario.

In [18]:
colunas_bow = [coluna for coluna in df_final.columns if coluna.startswith("bow_")]

frequencia_palavras = df_final[colunas_bow].sum().sort_values(ascending=False)

print(f"Total de palavras diferentes: {len(frequencia_palavras)}")

frequencia_palavras.head(10)

Total de palavras diferentes: 1495


bow_lbf           159
bow_basquete      125
bow_feminino       86
bow_liga           57
bow_paulista       48
bow_americana      46
bow_jogo           39
bow_campeonato     39
bow_santo          37
bow_selecao        35
dtype: int64

In [19]:
frequencia_palavras.tail(10)

bow_finalista      1
bow_finalissima    1
bow_xaras          1
bow_workshop       1
bow_wheeler        1
bow_washington     1
bow_vou            1
bow_votos          1
bow_voou           1
bow_voltou         1
dtype: int64

In [20]:
documentos_por_palavra = (df_final[colunas_bow] > 0).sum().sort_values(ascending=False)
documentos_por_palavra.index = documentos_por_palavra.index.str.replace("bow_", "", regex=False)

documentos_por_palavra.head(10)

lbf           158
basquete      119
feminino       86
liga           57
paulista       48
americana      46
jogo           39
campeonato     39
santo          37
selecao        35
dtype: int64

In [22]:
df_final["palavras_unicas"] = (df_final[colunas_bow] > 0).sum(axis=1)

df_final[["titulo_noticia", "palavras_unicas"]].sort_values("palavras_unicas", ascending=True).head(10)

,titulo_noticia,palavras_unicas
518,tudo_igual 2,0
525,32047,0
185,e meu,0
262,14849 2,0
798,mais um,0
905,novo_capitulo,0
850,9166 2,0
152,100paio,0
29,segue_vivo,0
526,de ponta a ponta 4,1
